# ETL Data Loading Validation
## Verify that data was correctly loaded into the database

**Purpose:** Count data before and after ETL, compare to verify correctness

**Sections:**
1. YOLO ETL Validation
2. LLM ETL Validation  
3. Clustering ETL Validation

**How to use:**
- Set input paths and parameters in each section
- Run cells to count source data
- Compare with database counts
- Check if numbers match ✅ or differ ❌

---
## Setup & Helper Functions

In [ ]:
import os
import pandas as pd
import pickle
import psycopg2
from pathlib import Path
from dotenv import load_dotenv

from source import image_id_converter as img_idc


database_name = 'image_analysis_dev'
#database_name = 'image_analysis_test'

db_env = '.env'
#db_env = '.env.test'

# Load test environment (override=True ensures it replaces any previously loaded values)
load_dotenv(db_env, override=True)

# Explicit check — you see exactly what you're connecting to
db_name = os.getenv('DB_NAME')
print(f"Connecting to: {db_name}")
assert db_name == database_name, f"Wrong database loaded: {db_name}"


print("✅ Libraries loaded")

In [ ]:
def count_image_files(folder_path, extension='.tif'):
    """Count image files with given extension in folder."""
    path = Path(folder_path)
    files = list(path.glob(f'*{extension}'))
    return len(files)

def count_csv_rows(csv_path):
    """Count rows in CSV (excluding header)."""
    df = pd.read_csv(csv_path)
    return len(df)

def count_ground_truth_labels(csv_path, label_columns):
    """Count total ground truth labels (rows * number of label columns)."""
    df = pd.read_csv(csv_path)
    n_rows = len(df)
    n_labels = len(label_columns)
    return n_rows * n_labels

def count_ground_truth_labels_select(csv_path, folder_path, label_columns):
    """Count total ground truth labels (rows * number of label columns)."""
    label_data = pd.read_csv(csv_path)
    img_ids = list(label_data.image_id)
    label_data['image_id'] = img_idc.reconvert_image_ids(img_ids)

    # Extract IDs from filenames
    image_files = list(folder_path.glob('*.tif'))
    source_ids = [str(''.join(filter(str.isdigit, f.stem))) for f in image_files if any(c.isdigit() for c in f.stem)]
    # print('source_ids:')
    # print(source_ids)
    # print(type(source_ids))
    # # Filter dataframe
    # print(len(label_data['image_id']))
    # print(type(label_data['image_id']))
    
    label_data_filtered = label_data[label_data['image_id'].isin(source_ids)]
    # selection_bools = []
    # for index, row in label_data.iterrows():
    #     selection_bools.append(int(row.image_id) in source_ids)
# 
    # label_data_filtered = label_data[selection_bools]

    # print("label_data['image_id'].isin(source_ids):")
    # print(len(label_data['image_id'].isin(source_ids)))
    # print(type(label_data['image_id'].isin(source_ids)))
    # print(label_data['image_id'].isin(source_ids)[0:3])
# 
    # print('label_data_filtered:')
    # print(label_data_filtered)
    
    df = label_data_filtered
    n_rows = len(df)
    n_labels = len(label_columns)
    return n_rows * n_labels



def query_database(query):
    """Execute query and return single count result."""
    conn = psycopg2.connect(
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD', ''),
        host=os.getenv('DB_HOST'),
        port=os.getenv('DB_PORT')
    )
    cur = conn.cursor()
    cur.execute(query)
    result = cur.fetchone()[0]
    cur.close()
    conn.close()
    return result

def compare_counts(source_count, db_count, description):
    """Compare source and database counts, print result."""
    match = "✅" if source_count == db_count else "❌"
    print(f"{match} {description}")
    print(f"   Source: {source_count}")
    print(f"   Database: {db_count}")
    if source_count != db_count:
        diff = db_count - source_count
        print(f"   Difference: {diff:+d}")
    print()

print("✅ Helper functions defined")

---
## Section 1: YOLO ETL Validation

### 1.1 Set YOLO Input Paths

In [ ]:
project_path = Path.cwd()
#root_path = (project_path / '..' / 'data_archive').resolve()
root_path = (project_path / '..' / 'data_folders').resolve()
# data_path = root_path / 'filter_out_people_multi_approach_ubelix_2025.10.26'
# data_path = root_path /'restructured/filter_out_people_multi_approach_ubelix_2025.10.26'
# data_path = root_path / 'test_filter_out_people_multi_approach_ubelix_2026.03.09'
data_path = root_path / 'from_ubelix/people_detect_multiapproach_20260309_214342'

tif_data_path = root_path / 'data_1'


In [ ]:

files = sorted(os.listdir(data_path), key=lambda f: os.path.getmtime(os.path.join(data_path, f)), reverse=True)
files

In [ ]:
# ========== EDIT THESE PATHS ==========
#yolo_image_folder = '../data_folders/data'  # Folder with .tif images
yolo_image_folder = tif_data_path
yolo_image_extension = '.tif'  # Image file extension
#yolo_ground_truth_csv = '../data_folders/labels_mod.csv'  # Ground truth CSV
yolo_ground_truth_csv = data_path / 'labels_mod.csv'
#yolo_results_csv = '../data_folders/yolo_results.csv'  # YOLO predictions CSV
#yolo_results_csv = data_path / 'people_detect_multi_approach_labels_results_yolo_20260213_144223.csv'
#yolo_results_csv = data_path /'people_detect_multi_approach_labels_results_yolo_20251026_144027.csv'
yolo_results_csv = data_path /'people_detect_multi_approach_labels_results_yolo_20260309_214342.csv'
yolo_source = 'giub'  # Source identifier used in database
#yolo_analysis_run_timestamp = '20260213_144223'  # Timestamp of this run
#yolo_analysis_run_timestamp = '20251026_144027'
yolo_analysis_run_timestamp = '20260309_214342'
# ======================================

# Label columns in ground truth CSV (excluding image_id)
yolo_label_columns = ['with_person', 'person_recognisable', 'is_photo', 
                      'with_church', 'in_high_alpine_environment']

print(f"✅ YOLO paths configured for source: {yolo_source}")


### 1.2 Count YOLO Source Data

In [ ]:
print("📊 Counting YOLO source data...\n")

# Count images
yolo_n_images = count_image_files(yolo_image_folder, yolo_image_extension)
print(f"Images: {yolo_n_images}")

# Count ground truth labels
yolo_n_gt_labels = count_ground_truth_labels(yolo_ground_truth_csv, yolo_label_columns)
print(f"Ground truth labels: {yolo_n_gt_labels}")

yolo_n_gt_labels = count_ground_truth_labels_select(yolo_ground_truth_csv, yolo_image_folder, yolo_label_columns)
print(f"Ground truth labels: {yolo_n_gt_labels}")

# Count predictions (one prediction per row in results CSV)
yolo_n_predictions = count_csv_rows(yolo_results_csv)
print(f"Predictions: {yolo_n_predictions}")

print("\n✅ YOLO source counts complete")

### 1.3 Query YOLO Database Counts

In [ ]:
print("📊 Querying YOLO database counts...\n")

# Images for this source
yolo_db_images = query_database(f"""
    SELECT COUNT(*) FROM images WHERE source = '{yolo_source}'
""")

# Ground truth labels for these images (current only)
yolo_db_gt = query_database(f"""
    SELECT COUNT(*) 
    FROM ground_truth_history gt
    JOIN images i ON gt.image_id = i.image_id
    WHERE i.source = '{yolo_source}' AND gt.is_current = TRUE
""")

# Predictions from this analysis run
yolo_db_predictions = query_database(f"""
    SELECT COUNT(*)
    FROM predictions p
    JOIN analysis_runs ar ON p.analysis_run_id = ar.analysis_run_id
    WHERE ar.run_timestamp = '{yolo_analysis_run_timestamp}'
""")

# Analysis runs
yolo_db_runs = query_database(f"""
    SELECT COUNT(*) FROM analysis_runs 
    WHERE run_timestamp = '{yolo_analysis_run_timestamp}'
""")

print("✅ YOLO database counts complete")

### 1.4 Compare YOLO Counts

In [ ]:
print("="*50)
print("YOLO ETL VALIDATION RESULTS")
print("="*50 + "\n")

compare_counts(yolo_n_images, yolo_db_images, "Images")
compare_counts(yolo_n_gt_labels, yolo_db_gt, "Ground Truth Labels")
compare_counts(yolo_n_predictions, yolo_db_predictions, "Predictions")
compare_counts(1, yolo_db_runs, "Analysis Runs (expected: 1)")

---
## Section 2: LLM ETL Validation

### 2.1 Set LLM Input Paths

In [ ]:
project_path = Path.cwd()
# root_path = (project_path / '..' / 'data_archive').resolve()
root_path = (project_path / '..' / 'data_folders').resolve()
# data_path = root_path / 'filter_out_people_multi_approach_ubelix_2025.10.26'
# data_path = root_path /'restructured/filter_out_people_multi_approach_ubelix_2025.10.26'
# data_path = root_path /'restructured/LLM_multi_object_struct_minicpm_ubelix_2025.10.25'
# data_path = root_path / 'test_filter_out_people_multi_approach_ubelix_2026.03.09'
data_path = root_path / 'from_ubelix/people_detect_multiapproach_20260309_214410'
# data_path = root_path / 'from_ubelix/multi_object_struct_minicpm_20260309_174756'
tif_data_path = root_path  / 'data_1'

In [ ]:

files = sorted(os.listdir(data_path), key=lambda f: os.path.getmtime(os.path.join(data_path, f)), reverse=True)
files

In [ ]:
# ========== EDIT THESE PATHS ==========
# llm_image_folder = '../data_folders/data'  # Folder with .tif images
llm_image_folder = tif_data_path  # Folder with .tif images
llm_image_extension = '.tif'  # Image file extension
# llm_ground_truth_csv = '../data_folders/labels_mod.csv'  # Ground truth CSV
llm_ground_truth_csv = data_path / 'labels_mod.csv'  # Ground truth CSV
# llm_results_pickle = '../data_folders/results_llm_20260213_144241.pkl'  # LLM results pickle
# llm_results_pickle = data_path / 'results_llm_people_detect_multi_approach_20260213_144241.pkl'  # LLM results pickle
# llm_results_pickle = data_path / 'results_multi_object_struct_minicpm_20251025_152957.pkl'
# llm_results_pickle = data_path / 'results_multi_object_struct_minicpm_20260309_174756.pkl'
llm_results_pickle = data_path / 'results_llm_people_detect_multi_approach_20260309_214410.pkl'
llm_source = 'giub'  # Source identifier
#llm_analysis_run_timestamp = '20260213_144241'  # Timestamp (from pickle filename)
#llm_analysis_run_timestamp = '20251025_152957'
#llm_analysis_run_timestamp = '20260309_174756'
llm_analysis_run_timestamp = '20260309_214410'
# ======================================

# Label columns (same as YOLO)
llm_label_columns = ['with_person', 'person_recognisable', 'is_photo', 
                     'with_church', 'in_high_alpine_environment']

print(f"✅ LLM paths configured for source: {llm_source}")


### 2.2 Count LLM Source Data

In [ ]:
print("📊 Counting LLM source data...\n")

# Count images
llm_n_images = count_image_files(llm_image_folder, llm_image_extension)
print(f"Images: {llm_n_images}")

# Count ground truth labels
llm_n_gt_labels = count_ground_truth_labels(llm_ground_truth_csv, llm_label_columns)
print(f"Ground truth labels: {llm_n_gt_labels}")

llm_n_gt_labels = count_ground_truth_labels_select(llm_ground_truth_csv, llm_image_folder, llm_label_columns)
print(f"Ground truth labels: {yolo_n_gt_labels}")

# Load pickle and count predictions
with open(llm_results_pickle, 'rb') as f:
    llm_results = pickle.load(f)

# Extract predictions dataframe from nested dictionary structure
# Structure: {'timestamp': {'predictions': {'label_name': dataframe}}}
timestamp_key = list(llm_results.keys())[0]
predictions_dict = llm_results[timestamp_key]['predictions']

# Count total predictions (sum of all rows across all label dataframes)
llm_n_predictions = sum(len(df) for df in predictions_dict.values())
print(f"Predictions: {llm_n_predictions}")

# Count LLM responses (one per image)
llm_n_responses = llm_n_images  # Should match number of images
print(f"LLM responses: {llm_n_responses}")

print("\n✅ LLM source counts complete")

### 2.3 Query LLM Database Counts

In [ ]:
print("📊 Querying LLM database counts...\n")

# Images (may be same as YOLO if same source)
llm_db_images = query_database(f"""
    SELECT COUNT(*) FROM images WHERE source = '{llm_source}'
""")

# Ground truth labels
llm_db_gt = query_database(f"""
    SELECT COUNT(*) 
    FROM ground_truth_history gt
    JOIN images i ON gt.image_id = i.image_id
    WHERE i.source = '{llm_source}' AND gt.is_current = TRUE
""")

# Predictions from this run
llm_db_predictions = query_database(f"""
    SELECT COUNT(*)
    FROM predictions p
    JOIN analysis_runs ar ON p.analysis_run_id = ar.analysis_run_id
    WHERE ar.run_timestamp = '{llm_analysis_run_timestamp}'
""")

# LLM responses from this run
llm_db_responses = query_database(f"""
    SELECT COUNT(*)
    FROM llm_responses lr
    JOIN analysis_runs ar ON lr.analysis_run_id = ar.analysis_run_id
    WHERE ar.run_timestamp = '{llm_analysis_run_timestamp}'
""")

# Analysis runs
llm_db_runs = query_database(f"""
    SELECT COUNT(*) FROM analysis_runs 
    WHERE run_timestamp = '{llm_analysis_run_timestamp}'
""")

print("✅ LLM database counts complete")

### 2.4 Compare LLM Counts

In [ ]:
print("="*50)
print("LLM ETL VALIDATION RESULTS")
print("="*50 + "\n")

compare_counts(llm_n_images, llm_db_images, "Images")
compare_counts(llm_n_gt_labels, llm_db_gt, "Ground Truth Labels")
compare_counts(llm_n_predictions, llm_db_predictions, "Predictions")
compare_counts(llm_n_responses, llm_db_responses, "LLM Responses")
compare_counts(1, llm_db_runs, "Analysis Runs (expected: 1)")

---
## Section 3: Clustering ETL Validation

### 3.1 Set Clustering Input Paths

In [ ]:
project_path = Path.cwd()

#visual_genome_path = (project_path/ '..' /'data_folders' / 'visual_genome_data').resolve()
#visual_genome_proc_path = (project_path/ '..' /'data_folders' / 'visual_genome_proc_data').resolve()

data_path_comb = (project_path/ '..' /'data_folders'/'combined_data').resolve()


In [ ]:
#files = sorted(os.listdir(visual_genome_path), key=lambda f: os.path.getmtime(os.path.join(visual_genome_path, f)), reverse=True)
#files[0:7]
files = sorted(os.listdir(data_path_comb), key=lambda f: os.path.getmtime(os.path.join(data_path_comb, f)), reverse=True)
files[0:7]


In [ ]:
# ========== EDIT THESE PATHS ==========
# clustering_image_paths_csv = '../visual_genome_proc/labels.csv'  # CSV with file_paths column
# clustering_image_paths_csv = visual_genome_proc_path / 'labels.csv'  # CSV with file_paths column
clustering_image_paths_csv = data_path_comb / 'labels.csv' 
# clustering_ground_truth_csv = '../visual_genome_proc/labels.csv'  # Ground truth CSV (can be same file)
# clustering_ground_truth_csv = visual_genome_proc_path / 'labels.csv'  # Ground truth CSV (can be same file)
clustering_ground_truth_csv = data_path_comb / 'labels.csv'

# clustering_results_pickle = '../visual_genome_proc/'metadata_results_clustering.pkl'  # Clustering results
# clustering_results_pickle = visual_genome_path / 'results_clustering_pipeline_20260208_225215.pkl'
# clustering_results_pickle = data_path_comb / 'results_clustering_pipeline_20260312_120235.pkl'
clustering_results_pickle = data_path_comb / 'results_clustering_pipeline_20260314_211803.pkl'


#clustering_source = 'visual_genome'  # Source identifier

clustering_source_1 = 'visual_genome' 
clustering_source_2 = 'swiss_topo'
#clustering_analysis_run_timestamp = '2026-02-08 22:52:15.995956'  # Timestamp of this run
# clustering_analysis_run_timestamp = '20260208_225215'
# clustering_analysis_run_timestamp = '20260312_120235'
clustering_analysis_run_timestamp = '20260314_211803'

# ======================================

# print(f"✅ Clustering paths configured for source: {clustering_source}")


### 3.2 Count Clustering Source Data

In [ ]:
print("📊 Counting clustering source data...\n")

# Count images from file_paths CSV
image_paths_df = pd.read_csv(clustering_image_paths_csv)
clustering_n_images = len(image_paths_df)
print(f"Images: {clustering_n_images}")

# Count ground truth labels from CSV
gt_df = pd.read_csv(clustering_ground_truth_csv)
# Count non-image_id columns as labels
label_columns = [col for col in gt_df.columns if (col != 'image_id') and (col != 'file_paths') and (col != 'file_source')]
clustering_n_gt_labels = len(gt_df) * len(label_columns)
print(f"Ground truth labels: {clustering_n_gt_labels}")

# Load clustering results from pickle
with open(clustering_results_pickle, 'rb') as f:
    clustering_metadata = pickle.load(f)

# Extract cluster_data dataframe
cluster_data_df = clustering_metadata['cluster_data']
clustering_n_clusters = len(cluster_data_df)
print(f"Cluster assignments: {clustering_n_clusters}")

print("\n✅ Clustering source counts complete")

In [ ]:
image_paths_df.shape

In [ ]:
gt_df.head()

In [ ]:
label_columns

In [ ]:
clustering_analysis_run_timestamp

### 3.3 Query Clustering Database Counts

In [ ]:
print("📊 Querying clustering database counts...\n")

# Images for this source
clustering_db_images_1 = query_database(f"""
    SELECT COUNT(*) FROM images WHERE source = '{clustering_source_1}'
""")

clustering_db_images_2 = query_database(f"""
    SELECT COUNT(*) FROM images WHERE source = '{clustering_source_2}'
""")

clustering_db_images = clustering_db_images_1 + clustering_db_images_2

# Ground truth labels
clustering_db_gt_1 = query_database(f"""
    SELECT COUNT(*) 
    FROM ground_truth_history gt
    JOIN images i ON gt.image_id = i.image_id
    WHERE i.source = '{clustering_source_1}' AND gt.is_current = TRUE
""")

clustering_db_gt_2 = query_database(f"""
    SELECT COUNT(*) 
    FROM ground_truth_history gt
    JOIN images i ON gt.image_id = i.image_id
    WHERE i.source = '{clustering_source_2}' AND gt.is_current = TRUE
""")

clustering_db_gt = clustering_db_gt_1 + clustering_db_gt_2

# Clustering results from this run
clustering_db_clusters = query_database(f"""
    SELECT COUNT(*)
    FROM clustering_results cr
    JOIN analysis_runs ar ON cr.analysis_run_id = ar.analysis_run_id
    WHERE ar.run_timestamp = '{clustering_analysis_run_timestamp}'
""")

# Analysis runs
clustering_db_runs = query_database(f"""
    SELECT COUNT(*) FROM analysis_runs 
    WHERE run_timestamp = '{clustering_analysis_run_timestamp}'
""")

print("✅ Clustering database counts complete")

### 3.4 Compare Clustering Counts

In [ ]:
print("="*50)
print("CLUSTERING ETL VALIDATION RESULTS")
print("="*50 + "\n")

compare_counts(clustering_n_images, clustering_db_images, "Images")
compare_counts(clustering_n_gt_labels, clustering_db_gt, "Ground Truth Labels")
compare_counts(clustering_n_clusters, clustering_db_clusters, "Cluster Assignments")
compare_counts(1, clustering_db_runs, "Analysis Runs (expected: 1)")

---
## Summary

**How to interpret results:**

✅ = Count matches (data loaded correctly)  
❌ = Count differs (investigate discrepancy)

**Common reasons for mismatches:**
- Images already existed in database (from previous run)
- Ground truth CSV has different structure than expected
- Wrong analysis_run_timestamp specified
- Data was partially loaded (some images skipped)

**Next steps if validation fails:**
1. Check the analysis_run_timestamp is correct
2. Verify source identifier matches what was used in ETL
3. Query database directly to investigate: `psql image_analysis_dev`